# Modèle de langage et génération de séquence — Transformeurs

Dans ces travaux pratiques, nous allons étudier les différentes manières existantes pour décoder du texte étant donné un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage, dont la traduction, le question/réponse et la génération.

Nous allons prendre l'exemple de la génération de texte : elle passait obligatoirement par des textes à trou plus ou moins sophistiqués jusqu'à très récemment. L'arrivée de modèles puissants entraînés sur de grandes quantités de données a changé la donne : il est maintenant possible d'utiliser des méthodes neuronales.

Dans ce notebook, nous allons voir comment générer du texte à partir d'un modèle entraîné, en partant de la méthode la plus simple pour progresser jusqu'aux méthodes état de l'art. Cela vous permettra d'asseoir solidement votre compréhension des mécanismes liés aux réseaux de neurones appliqués au texte : l'utilisation de l'activation softmax en particulier, qui est au cœur des principales architectures NLP état de l'art.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de blog sur le sujet de Hugging Face peut servir de ressource complémentaire à ce notebook.

## Installation

La librairie [`transformers`](https://github.com/huggingface/transformers) développée par la startup franco-américaine [Hugging Face](https://huggingface.co/) met à disposition un grand nombre de modèles de la famille des transformeurs.

Elle s'installe facilement avec `pip` :

In [1]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 4.2 MB 5.2 MB/s 
     |████████████████████████████████| 596 kB 41.1 MB/s 
     |████████████████████████████████| 86 kB 3.6 MB/s 
     |████████████████████████████████| 6.6 MB 37.6 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles préentraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [2]:
#lang = "fr"
lang = "en"

## Import d'un modèle entraîné

En plus de mettre à disposition le code source de nombreuses architectures de transformeurs, la librairie `transformers` permet de récupérer de [nombreux modèles entraînés](https://huggingface.co/transformers/pretrained_models.html).

Pour ce TP, nous allons utiliser l'architecture GPT2, derrière le [modèle qui a surpris la communauté NLP](https://openai.com/blog/better-language-models/) par la qualité de ses générations (et qui a depuis vu son successeur, GPT3, atteindre [des résultats encore plus bluffants](https://github.com/elyase/awesome-gpt3)).

Pour cela il suffit d'utiliser la méthode `from_pretrained` sur la classe de l'architecture du modèle qui nous intéresse, avec comme argument le nom de l'entrainement que l'on souhaite récupérer. On peut récupérer le tokeniseur lié au modèle de la même manière.

In [3]:
import functools
import typing

import numpy
import tensorflow
import transformers
import tqdm.notebook


pretraining_name = "antoiloui/belgpt2" if lang == "fr" else "gpt2"

model = transformers.TFGPT2LMHeadModel.from_pretrained(pretraining_name)
tokenizer = transformers.GPT2Tokenizer.from_pretrained(pretraining_name)

Downloading:   0%|          | 0.00/665 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/475M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFGPT2LMHeadModel.

All the layers of TFGPT2LMHeadModel were initialized from the model checkpoint at gpt2.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Downloading:   0%|          | 0.00/0.99M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

## Utilisation du GPU

Par défaut, TensorFlow utilise les GPUs disponibles. Vérifions si Colab nous en a mis à disposition :

In [4]:
print(tensorflow.config.experimental.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques.

Pour cela, dans la librairie `transformers`, on peut vérifier sa config.

In [5]:
print(model.config)

GPT2Config {
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.19.4",
  "use_cache": true,
  "vocab_size": 50257
}



- Il possède 12 couches
- Son principal mécanisme de régularisation est le dropout
- Ses mécanismes de stabilisation de l'apprentissage visibles depuis l'architecture montrée ci-dessus sont la layer normalization et les connexions résiduelles.
- On peut lire sa valeur dans les clefs `bos_token_id` et `eos_token_id` de la config : `50256`.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on fonctionne de manière itérative : on génère les mots un par un, en commençant avec un texte vide ou une phrase que l'on souhaite compléter.

Pour générer un mot, on donne en entrée du réseau les mots générés jusque là(transformés en indices d'embedding). Le réseau produit alors un score par mot du vocabulaire. On choisit l'un de ces mots avant de boucler (concaténer ce mot à ce qui avait été généré puis donner le résultat en entrée au réseau pour générer le mot suivant).

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \Pi_{i=1}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

Tout l'enjeu pendant du decoding est de trouver la valeur maximale du produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approximer.

## Tokenisation

Le tokeniseur récupéré en même temps que le modèle permet de transformer une phrase d'entrée en index d'embedding que le modèle peut comprendre.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes qui transforment des chaînes de caractères en index de vocabulaire que le modèle comprend (ils sont les index de son module d'embedding).

In [6]:
def encode(sentence: str) -> tensorflow.Tensor:
  tokens = tokenizer.encode(sentence,
                            add_special_tokens=False,
                            return_tensors="tf")
  bos = tensorflow.constant([[model.config.bos_token_id]], dtype="int64")
  return tensorflow.concat([bos, tensorflow.cast(tokens, "int64")], axis=1)


def decode(tokens: tensorflow.Tensor) -> str:
  if tokens.ndim == 2:
    tokens = tensorflow.squeeze(tokens)
  return tokenizer.decode(tokens[1:], clean_up_tokenization_spaces=True)

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain mot. Étudions son comportement.

Dans Keras, pour calculer la passe forward d'un modèle, on utilise directement `model(input_tensor)` comme nous l'avons déjà vu.

Notez l'argument `past` : il permet d'éviter de recalculer des valeurs calculées pendant l'étape de decoding précédente. Si on donne l'argument past au modèle, il ne faut pas repasser les anciennes valeurs.

In [7]:
def describe_shapes(input_string: str,
                    output_tensor: tensorflow.Tensor,
                    output_past: typing.List[tensorflow.Tensor]
                   ) -> str:
  print("-" * 80)
  print(f"Taille de l'output pour l'input '{input_string}' : "
        f"{output_tensor.shape}")
  print(f"Taille du passé : {len(output_past)}")
  print(f"Type du passé : {type(output_past[0])}")


empty_input = ""
descartes_input = ("Je pense, donc je suis"
                   if lang == "fr"
                   else "I think, therefore I am")

empty_output, empty_past = model(encode(empty_input),
                                 return_dict=False)
descartes_output, descartes_past = model(encode(descartes_input),
                                         return_dict=False)

describe_shapes(empty_input, empty_output, empty_past)
describe_shapes(descartes_input, descartes_output, descartes_past)

--------------------------------------------------------------------------------
Taille de l'output pour l'input '' : (1, 1, 50257)
Taille du passé : 12
Type du passé : <class 'tensorflow.python.framework.ops.EagerTensor'>
--------------------------------------------------------------------------------
Taille de l'output pour l'input 'I think, therefore I am' : (1, 7, 50257)
Taille du passé : 12
Type du passé : <class 'tensorflow.python.framework.ops.EagerTensor'>


## Fonction `forward` adaptée aux besoins du décodage

Étant donné les observations faites dans la section précédente, nous utiliserons la fonction `forward` suivante, qui rend un output adapté au décodage : seule la dernière colonne nous intéresse (la prédiction du mot suivant). Cette fonction `forward` implémente cette sélection.

In [8]:
def forward(tokens: tensorflow.Tensor,
            past: typing.Optional[typing.List[tensorflow.Tensor]] = None
           ) -> typing.Tuple[tensorflow.Tensor, typing.List[tensorflow.Tensor]]:
  logits, new_past = model(tokens, past=past, return_dict=False)
  return logits[:, -1, :], new_past

Notez bien que quand l'argument `past` est utilisé, il ne faut pas redonner les tokens qui ont déjà été calculés, seulement le nouveau token à décoder. Il faut donc utiliser au choix :

- `forward(all_tokens)`
- `forward(last_token, past)`

où `all_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7, 50257)` là où `last_token` serait de forme `(1, 1, 50257)`.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Encodage de tout ce qui a été produit jusqu'ici (entrées et sorties précédentes)
2. Production de l'indice du mot suivant le plus probable
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons le nombre de mots produits seulement dans ce TP)

L'étape 2 est l'étape où tout se joue. Nous allons pour l'instant coder tout le reste, afin de pouvoir nous concentrer sur l'étape 2 par la suite.

In [9]:
def decoding_loop(step_function) -> str:

  @functools.wraps(step_function)
  def wrapper(prompt, length, *step_args, **step_kwargs):
    token_ids = encode(prompt)
    past = None
    decoded = [token_ids]
    for i in tqdm.notebook.trange(length, desc="Mots", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded += [index[None, None]]
    decoded_tensor = tensorflow.concat(decoded, axis=1)
    return decode(decoded_tensor)

  return wrapper

## Décodage greedy

Le décodage greedy est la méthode la plus simple pour décoder du texte depuis un modèle appris.

À chaque étape du décodage (pour décoder chaque mot), il consiste à prendre le mot avec la plus forte probabilité.

Plus formellement, dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \Pi_{i=0}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

Le décodage greedy choisira $\text{Mot}_i$ pour maximer $P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un mot moins probable pour ensuite atteindre des mots très probables.

L'exemple suivant est donné dans le blog de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

On peut y voir qu'après avoir décodé «&nbsp;The&nbsp;», on choisit «&nbsp;nice&nbsp;» parce ce que c'est le mot avec la probabilité maximale pour le modèle et que le decoding complet «&nbsp;The nice woman&nbsp;» a une probabilité inférieure (0.20) au decoding que l'on aurait obtenu en choisissant «&nbsp;dog&nbsp;» («&nbsp;The dog has&nbsp;», probabilité 0.36).

In [10]:
@decoding_loop
def greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  return tensorflow.math.argmax(logits, axis=1)[0]


print(greedy("I went to the", 10))

Mots:   0%|          | 0/10 [00:00<?, ?it/s]

I went to the store and bought a new pair of shoes. I


## Test sur des inputs variés

In [11]:
if lang == "fr":
    inputs = ["Je suis allé au",
              "Je me sens",
              "Comment vas-tu",
              "Comment se fait-il que",
              "Ça va merci, tu devrais",
              "Donald Trump vient de tweeter :",
              "Le président Trump vient de démissionner !"]
else:
    inputs = ["I went to the",
              "I am feeling",
              "How do you",
              "How comes that",
              "I'm fine thank you, you should",
              "Donald Trump just tweeted :",
              "Le président Trump just quit !"]

In [12]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = []
  for i in tqdm.notebook.tqdm(inputs, desc="Prompts", leave=False):
    generations.append(function(i, *args, **kwargs))
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)

test_decoding(greedy, 50)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the store and bought a new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you know if you're a good candidate for a job?

The answer is simple: you're not.

You're not a good candidate for a job because you're not a good candidate for a job.

You're not a
————————————————————————————————————————————————————————————————————————————————

On constate que les réponses «&nbsp;bouclent&nbsp;» assez rapidement. C'est une faille très classique du décodage greedy.

## Sampling

Comme nous venons de le constater, les résultats du décodage greedy posent de nombreux problèmes. Ils bouclent rapidement et sont souvent très génériques. L'objet des méthodes à suivre est de palier ces déficiences.

La première approche que nous allons voir consiste à échantilloner à partir des résultats du modèle plutôt que de toujours choisir le résultat le plus probable.

In [13]:
@decoding_loop
def sampling(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  softmaxed = tensorflow.nn.log_softmax(logits)
  return tensorflow.random.categorical(softmaxed, 1)[0][0]

test_decoding(sampling, 50)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the Natural Okanagan Fisherman's Co. this year to find out more about Han Hookowski's company. While the resulting addition of a tantalizing crackpot frieze to my Short-Tongued Flask, so I might as well mention
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling too much memesthesia now after reading around this blog about how I've become more addicted to frameworks and hopefully I will write more about my experience. I think it is due to dearth of knowledge and resources to talk about how to develop these frameworks.
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you wage war? Terroristic Terrorism Fatalities in Afghanistan 13,063 Terrorists Fatalities in the UK 13,020 Terrorism Ratings Worldwide 20 September 35

## Top-k sampling

Une variation de l'échantillonage consiste à n'échantilloner que depuis les `k` résultats maximaux, pour éviter de générer un mot trop improbable (même si cela n'arrive que rarement).

In [14]:
@decoding_loop
def k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor:
  topk = tensorflow.math.top_k(logits, k=k)
  softmaxed = tensorflow.nn.log_softmax(topk.values)
  sampled = tensorflow.random.categorical(softmaxed, 1)[0][0]
  return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_sampling, 50, k=20)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the mall today to get a copy of your book. I had a copy from the last time I went. Now I have to see it!<|endoftext|>The first time this year I had to do some research before coming across it.

There are some
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling pretty bad about this post. It's been so long. I thought I'd start this post at 1:15pm and I was thinking I was going to post this here at 1:30pm. I actually think I've finished posting the last
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you feel about this new movie adaptation of the popular classic "Fiddler on the Roof"?

We've had some wonderful things from the past. "Fiddler on the Roof" (2004), one of the first big-budget movies, has been
—————————

## Top-p sampling

Une autre amélioration de l'échantillonage consiste à ne considérer que les `n` premiers outputs, dont la probabilité sommée dépasse `p`, et pas les suivants.

In [15]:
@decoding_loop
def k_p_sampling(logits: tensorflow.Tensor, k: int, p: float
                ) -> tensorflow.Tensor:
    topk = tensorflow.math.top_k(logits, k=k)
    softmaxed = tensorflow.nn.softmax(topk.values)
    current_proba_sum = 0
    j = 0
    while current_proba_sum < p:
      current_proba_sum += softmaxed[0, j]
      j += 1
    sampled = tensorflow.random.categorical(
        tensorflow.math.log(softmaxed[:, :j]), 1)[0][0]
    return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_p_sampling, length=100, k=20, p=0.85)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the library to get some books for my birthday and I was very surprised to see some of the books. I love to read, so I'm excited to read more books. I'm really excited about this book. I have never had a bad time with this book. I love it so much. I'm so excited to read more books.

This is a good book, and I have never read a book that wasn't written by the author. I have never liked a novel that I haven
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling pretty good about it, but I have been playing a lot of video games since I was 14 and I don't have much of a sense of what my gaming life is like. So I'm trying to figure out what I'm playing and how I feel about it.

I've been playing a lot of different games lately, mostly MMOs and RPGs. I've always liked a lot of them, but I think 

## Utilisation des fonctions de `transformers`

Dans le futur, si vous avez besoin de générer du texte, vous pourrez utiliser la fonction `generate` des modèles de la librarie `transformers`. Voici un exemple :

In [16]:
inputs = ["Kim jong Un resigns after a headache"]

In [18]:
def transformers_generate(prompt: str,
                          length: int,
                          temperature: float,
                          top_k: int,
                          top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(input_ids=tensorflow.cast(token_ids, "int32"),
                             max_length=len(token_ids) + length,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             do_sample=True,
                             num_return_sequences=1)
  return decode(tensorflow.cast(generated, "int64"))

test_decoding(transformers_generate, length=1000, temperature=1, top_k=30, top_p=0.97)

Prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


————————————————————————————————————————————————————————————————————————————————
Prompt : Kim jong Un resigns
Génération : Kim jong Un resigns as head of the country's ruling Communist party. Photo credit: Xinhua News Agency

SEOUL, March 6 (Xinhua) -- Kim Jong Un's political party is under pressure to step down, after a "very bitter experience" with a high-ranking member of the ruling Communist Party who was expelled from the party.

A parliamentary group has summoned Kim's two deputy vice-presidents, Jang Song Thaek and Park Geun-hye, and the party's vice-presidents, Hwang Kyo-ahn and Moon Jae-in, for questioning after reports of Kim's resignation last October.

The members of the parliamentary committee on the constitution said the former vice president was expelled because he tried to influence the party's leadership by taking part in anti-Korean protests during South Korean Independence Day in December.

The vice-presidents also accused Choi Soon-sil of attempting to take over the